#### Project 2 — CFPB Triage Latency Prediction

#### Split Design + Regression Baselines

#### Objective

Create a leakage-safe temporal train/validation/test split and establish
simple regression baselines before feature engineering or complex models.

#### Prediction event:- Complaint received.

#### Prediction timestamp:- `Date received`

#### Target:- `triage_delay_days`

#### Core principle

- The model must learn from historical complaints and be evaluated on
later complaints.The test set remains completely untouched until final evaluation.

In [1]:
## Imports and Paths
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent

DATA_PATH = PROJECT_ROOT / "Data" / "raw" / "complaints-cfpb-raw.csv"
PROCESSED_DIR = PROJECT_ROOT / "Data" / "processed"
REPORT_DIR = PROJECT_ROOT / "Reports" / "project2"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
## Load & Construct Target
df = pd.read_csv(DATA_PATH)

df["Date received"] = pd.to_datetime(
    df["Date received"],
    errors="coerce",
    utc=True
)

df["Date sent to company"] = pd.to_datetime(
    df["Date sent to company"],
    errors="coerce",
    utc=True
)

df["triage_delay_days"] = (
    df["Date sent to company"] - df["Date received"]
).dt.total_seconds() / (24 * 60 * 60)

df = df.sort_values("Date received").reset_index(drop=True)

print("Shape:", df.shape)
print("Date received range:")
print(df["Date received"].min(), "→", df["Date received"].max())

Shape: (81946, 17)
Date received range:
2026-03-01 00:41:56+00:00 → 2026-08-11 14:27:33+00:00


#### 1. Temporal Structure

Before choosing percentages for train/validation/test, inspect the
actual chronological coverage.

The split is not merely a mathematical partition.

It represents a deployment assumption:

> Train on historical complaints → tune on later complaints →
> evaluate on future complaints.

In [ ]:
## Monthly Volume
monthly = (
    df.groupby(df["Date received"].dt.to_period("M"))
      .size()
      .to_frame("complaint_count")
)

monthly["percentage"] = (
    monthly["complaint_count"] / len(df) * 100
)

display(monthly)

C:\Users\MY PC\AppData\Local\Temp\ipykernel_3088\2368122506.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df.groupby(df["Date received"].dt.to_period("M"))


,complaint_count,percentage
Date received,,
2026-03,21630,26.395431
2026-04,21048,25.685207
2026-05,17385,21.215190
2026-06,16721,20.404901
2026-07,5161,6.298050
2026-08,1,0.001220


#### 2. Temporal Split Design

#### Proposed split

Use approximately:

- 70% earliest observations → Train
- 15% following observations → Validation
- 15% latest observations → Test

The split is based on `Date received`, not random sampling.

#### Why?

At deployment time, we predict future complaints.

A temporal split therefore provides a more realistic estimate of
future generalization.

#### Important

- The exact dates are determined from the data rather than manually
invented.

In [4]:
n = len(df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
validation_df = df.iloc[train_end:validation_end].copy()
test_df = df.iloc[validation_end:].copy()

print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))

Train: 57362
Validation: 12292
Test: 12292


In [5]:
## Split Bounderies
split_summary = pd.DataFrame([
    {
        "split": "train",
        "rows": len(train_df),
        "start": train_df["Date received"].min(),
        "end": train_df["Date received"].max()
    },
    {
        "split": "validation",
        "rows": len(validation_df),
        "start": validation_df["Date received"].min(),
        "end": validation_df["Date received"].max()
    },
    {
        "split": "test",
        "rows": len(test_df),
        "start": test_df["Date received"].min(),
        "end": test_df["Date received"].max()
    }
])

display(split_summary)

,split,rows,start,end
0,train,57362,2026-03-01 00:41:56+00:00,2026-05-27 18:08:25+00:00
1,validation,12292,2026-05-27 18:09:09+00:00,2026-06-17 14:33:53+00:00
2,test,12292,2026-06-17 14:34:00+00:00,2026-08-11 14:27:33+00:00


In [ ]:
## Check Temporal Leakage
assert train_df["Date received"].max() < validation_df["Date received"].min() # assert checks condition
assert validation_df["Date received"].max() < test_df["Date received"].min()

print("Temporal ordering check: PASS") 

Temporal ordering check: PASS


In [7]:
## Target Distribution Across Splits
def target_summary(data, name):
    y = data["triage_delay_days"]

    return {
        "split": name,
        "count": len(y),
        "mean": y.mean(),
        "median": y.median(),
        "p90": y.quantile(0.90),
        "p95": y.quantile(0.95),
        "p99": y.quantile(0.99),
        "max": y.max()
    }

split_target_summary = pd.DataFrame([
    target_summary(train_df, "train"),
    target_summary(validation_df, "validation"),
    target_summary(test_df, "test")
])

display(split_target_summary)

,split,count,mean,median,p90,p95,p99,max
0,train,57362,2.218114,0.009861,0.045278,18.865344,46.728970,146.791586
1,validation,12292,1.098205,0.009942,0.037993,7.419414,27.991120,91.845995
2,test,12292,0.181871,0.010081,0.033240,0.055139,6.814138,54.981493


In [8]:
## Delay -Tail Stability
tail_thresholds = [0.05, 1, 3, 7, 14, 30, 60]

rows = []

for split_name, data in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df)
]:
    for threshold in tail_thresholds:
        rows.append({
            "split": split_name,
            "threshold_days": threshold,
            "count": int(
                (data["triage_delay_days"] > threshold).sum()
            ),
            "percentage": (
                data["triage_delay_days"] > threshold
            ).mean() * 100
        })

tail_stability = pd.DataFrame(rows)

display(tail_stability)

,split,threshold_days,count,percentage
0,train,0.05,5403,9.419128
1,train,1.00,4269,7.442209
2,train,3.00,4169,7.267878
3,train,7.00,3934,6.858199
4,train,14.00,3316,5.780831
5,train,30.00,2123,3.701056
6,train,60.00,123,0.214428
7,validation,0.05,959,7.801822
8,validation,1.00,712,5.792385
9,validation,3.00,699,5.686625


In [9]:
## Product Stability
product_split = pd.crosstab(
    df["Product"],
    pd.cut(
        df.index,
        bins=[-1, train_end - 1, validation_end - 1, len(df) - 1],
        labels=["train", "validation", "test"]
    ),
    normalize="columns"
) * 100

display(product_split.round(2))

col_0,train,validation,test
Product,,,
Checking or savings account,17.03,18.60,21.31
Credit card,16.58,16.95,18.11
Credit reporting or other personal consumer reports,3.86,4.79,4.63
Debt collection,34.26,31.95,27.77
Debt or credit management,0.90,0.67,0.98
"Money transfer, virtual currency, or money service",8.26,7.42,7.30
Mortgage,6.29,6.78,7.26
"Payday loan, title loan, personal loan, or advance loan",3.47,3.69,3.69
Prepaid card,1.05,1.23,1.28


In [10]:
## Company Stability
train_companies = set(train_df["Company"].dropna().unique())
validation_companies = set(validation_df["Company"].dropna().unique())
test_companies = set(test_df["Company"].dropna().unique())

company_coverage = {
    "train_unique_companies": len(train_companies),
    "validation_unique_companies": len(validation_companies),
    "test_unique_companies": len(test_companies),
    "validation_seen_in_train_pct": (
        len(validation_companies & train_companies)
        / len(validation_companies) * 100
    ),
    "test_seen_in_train_pct": (
        len(test_companies & train_companies)
        / len(test_companies) * 100
    )
}

display(pd.Series(company_coverage))

train_unique_companies          1773.000000
validation_unique_companies      888.000000
test_unique_companies            843.000000
validation_seen_in_train_pct      84.909910
test_seen_in_train_pct            87.663108
dtype: float64

#### 3. Split Freeze

Once the temporal split passes structural and coverage checks:

- do not move rows between splits
- do not optimize split dates for model performance
- do not inspect test predictions during model development
- do not tune hyperparameters using test results

The test set represents the final future-like evaluation.

In [11]:
train_df.to_csv(
    PROCESSED_DIR / "project2_train.csv",
    index=False
)

validation_df.to_csv(
    PROCESSED_DIR / "project2_validation.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR / "project2_test.csv",
    index=False
)

print("Temporal split files saved.")

Temporal split files saved.


#### BaseLine Metrics
#### 4. Regression Baselines

- Before feature engineering or sophisticated models, establish simple
baselines.

#### Baseline hierarchy

1. Mean prediction
2. Median prediction
3. Simple linear regression / Ridge using basic intake features

#### Why baselines matter

A model is useful only if it beats a simple alternative.

For this highly skewed target, the median baseline is particularly
important because the mean is heavily influenced by the long-delay tail.

#### Primary metric

MAE — Mean Absolute Error

Interpretation:

> On average, how many days is the prediction away from the actual
> triage delay?

#### Secondary metrics

- Median Absolute Error
- RMSE
- R²

- RMSE is retained because large errors in the long-delay tail may have
operational importance. 

In [12]:
### Evalution Helper
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    mean_squared_error,
    r2_score
)

def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "Median_AE": median_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

In [13]:
## Mean & Meadian Baselines
y_train = train_df["triage_delay_days"]
y_validation = validation_df["triage_delay_days"]

mean_prediction = y_train.mean()
median_prediction = y_train.median()

mean_pred = np.full(
    len(validation_df),
    mean_prediction
)

median_pred = np.full(
    len(validation_df),
    median_prediction
)

baseline_results = pd.DataFrame([
    {
        "model": "Train Mean",
        **regression_metrics(y_validation, mean_pred)
    },
    {
        "model": "Train Median",
        **regression_metrics(y_validation, median_pred)
    }
])

display(baseline_results)

,model,MAE,Median_AE,RMSE,R2
0,Train Mean,3.034631,2.209497,5.510240,-0.043087
1,Train Median,1.093132,0.005822,5.503912,-0.040692


#### Simple feature baseline
- For the first ML baseline, use only information that is clearly available
at intake.

- We'll intentionally keep this extremely simple.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

def add_basic_time_features(data):
    result = data.copy()

    result["received_hour"] = result["Date received"].dt.hour
    result["received_dayofweek"] = result["Date received"].dt.dayofweek
    result["received_month"] = result["Date received"].dt.month
    result["received_day"] = result["Date received"].dt.day

    return result

train_basic = add_basic_time_features(train_df)
validation_basic = add_basic_time_features(validation_df)

numeric_features = [
    "received_hour",
    "received_dayofweek",
    "received_month",
    "received_day"
]

categorical_features = [
    "Product",
    "Company",
    "State"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_features
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            categorical_features
        )
    ]
)

ridge_baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

ridge_baseline.fit(
    train_basic[numeric_features + categorical_features],
    y_train
)

ridge_pred = ridge_baseline.predict(
    validation_basic[numeric_features + categorical_features]
)

ridge_results = pd.DataFrame([
    {
        "model": "Ridge Basic Intake Features",
        **regression_metrics(y_validation, ridge_pred)
    }
])

display(ridge_results)

,model,MAE,Median_AE,RMSE,R2
0,Ridge Basic Intake Features,3.030422,1.775747,5.813598,-0.161099


In [15]:
## Compare Baseline
all_baselines = pd.concat(
    [
        baseline_results,
        ridge_results
    ],
    ignore_index=True
).sort_values("MAE")

display(all_baselines)

,model,MAE,Median_AE,RMSE,R2
1,Train Median,1.093132,0.005822,5.503912,-0.040692
2,Ridge Basic Intake Features,3.030422,1.775747,5.813598,-0.161099
0,Train Mean,3.034631,2.209497,5.510240,-0.043087


In [16]:
### Tail-Specific baseline Evaluation
validation_analysis = validation_df[
    [
        "Complaint ID",
        "triage_delay_days"
    ]
].copy()

validation_analysis["mean_prediction"] = mean_pred
validation_analysis["median_prediction"] = median_pred
validation_analysis["ridge_prediction"] = ridge_pred

validation_analysis["ridge_absolute_error"] = (
    validation_analysis["triage_delay_days"]
    - validation_analysis["ridge_prediction"]
).abs()

validation_analysis["delay_over_1_day"] = (
    validation_analysis["triage_delay_days"] > 1
)

validation_analysis["delay_over_7_days"] = (
    validation_analysis["triage_delay_days"] > 7
)

display(
    validation_analysis.groupby("delay_over_1_day")[
        "ridge_absolute_error"
    ].agg(
        count="size",
        mean="mean",
        median="median",
        p90=lambda x: x.quantile(0.90)
    )
)

,count,mean,median,p90
delay_over_1_day,,,,
False,11580,2.305978,1.679641,4.269867
True,712,14.812806,10.974460,33.397885


#### Final Analysis Insights

#### Split

The problem is treated as a temporal forecasting-style regression task:

Historical complaints → Train
Later complaints → Validation
Future complaints → Test

- This better represents deployment than a random split.

#### Target

The target remains:- `triage_delay_days`

- No target clipping or removal of long delays has been performed.

#### Baseline philosophy

- The mean and median baselines establish the minimum standard a learned
model must beat.

- The median baseline is especially important because the target is highly
right-skewed.

#### Evaluation

- MAE is the primary metric because it is directly interpretable in days
and is less dominated by extreme observations than RMSE.

- RMSE remains useful for understanding large errors.

#### Tail

- Long-delay cases must be evaluated separately.

- A model could achieve an attractive overall MAE while performing poorly
on the operationally important delayed cases.

#### Feature baseline

- The first learned baseline intentionally uses only simple intake-time
features.

- No complex feature engineering or target transformation has been
introduced.

#### Test

- The test set is now frozen and must not be used for model selection.